# 🧪 Lab 1 — Introduction, Image Formation & Camera Geometry

| | |
|---|---|
| **Sessions** | 1 – 5 (Intro to CV · Image formation · Pinhole camera · Camera geometry & calibration) |
| **Weeks** | 1 – 2 |
| **Learning Outcomes** | **LO 1.1** Pinhole camera model & image formation · **LO 1.2** Camera geometry & calibration |
| **Type** | Practice assignment — do it **before** your Lab workshop (Session 5) |
| **Where to submit** | This notebook (`Lab1_Image_Formation_and_Camera_Geometry.ipynb`) on Canvas |
| **Estimated time** | 90 – 120 minutes |

---

### 🎯 Objectives
By the end of this lab you will be able to:
1. Prepare a working **Python + OpenCV environment** for a computer-vision project.
2. Treat an image as a **2-D numeric array** and manipulate it (brightness, inversion, channels).
3. Reproduce the **pinhole camera / perspective projection** math $x=f\frac{X}{Z}$ and explain its effects (size scaling, vanishing points, inversion).
4. Combine the **extrinsic** $[R\mid t]$ and **intrinsic** $K$ matrices into the full projection $\mathbf{x}=K\,[R\mid t]\,\mathbf{X}$.
5. Run a **camera calibration** (DLT + `cv2.calibrateCamera`) on synthetic checkerboards and compare with the ground-truth intrinsics.
6. Visualise and correct **lens distortion**.

> 💡 **How to work with this notebook:** read the short recap, then solve each
> **TASK**. Code cells marked with `# ==== YOUR CODE HERE ====` are the parts we
> will grade. Answer conceptual questions directly in the markdown cell below
> the question. Keep all outputs visible before exporting/submitting.



## Part 0 · Prepare your environment 

The very first thing a vision/data-science project needs is a **reproducible
environment**. Run the cell below, then fill in the table in **Task 0**.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
np.random.seed(42)   # reproducibility

print("✅ numpy   ", np.__version__)
print("✅ opencv  ", cv2.__version__)
print("✅ matplotlib", plt.matplotlib.__version__)

# quick sanity check that OpenCV really works on an image
probe = np.zeros((40, 60), dtype=np.uint8)
cv2.circle(probe, (30, 20), 12, 255, -1)
print("✅ OpenCV draws a circle:", probe.sum() > 0)
print("\nYour environment is ready. 🎉")



### 📝 Task 0 — Environment notes
1. I am using a Python virtual environment with `pip` to install and isolate the packages for this project.
2. A requirements file records exact dependency versions, so the same environment can be reproduced on another computer.
3. For camera coordinates $(X_c,Y_c,Z_c)$: $x=fX_c/Z_c$, $y=fY_c/Z_c$; in pixels, $u=f_xX_c/Z_c+c_x$, $v=f_yY_c/Z_c+c_y$.


## Part 1 · An image is just a number grid 📊

**Recap.** A digital image is a function $I(x,y)$ sampled on a grid and stored as a
2-D array. A *greyscale* image is one matrix of intensities $I\in[0,255]$; a *colour*
image is a stack of three channels (Blue, Green, Red in OpenCV). Operations such as
brightening/inverting are **point-wise** transforms $I'\leftarrow g(I)$.

The cell below builds a tiny synthetic "scene" image from geometric primitives.
Run it and inspect the array statistics.



In [ ]:
H, W = 200, 300
scene = np.full((H, W), 120, dtype=np.uint8)          # grey background
cv2.rectangle(scene, (40, 40), (140, 160), 200, -1)   # bright rectangle  = 'house'
cv2.circle(scene, (230, 80), 45, 30, -1)              # dark circle        = 'sun'
cv2.line(scene, (0, 180), (300, 140), 0, 6)           # black line         = 'road'

print("shape:", scene.shape, "| dtype:", scene.dtype)
print("min:", scene.min(), "| max:", scene.max(), "| mean: %.2f" % scene.mean())

plt.imshow(scene, cmap='gray', vmin=0, vmax=255)
plt.title("Synthetic scene I(x, y)"); plt.axis('off'); plt.show()



### 🛠️ Task 1 — Point-wise transforms 
Complete the two functions below.

* **Inversion** (photographic negative): $I' = 255 - I$.
* **Brightness shift**: $I' = \mathrm{clip}(I + b,\ 0,\ 255)$.

Then run the verification cell and make sure **both assertions pass**.



In [ ]:
def invert_image(img):
    """Return the photographic negative of img (uint8)."""
    return 255 - img


def shift_brightness(img, b):
    """Return img whose brightness is shifted by b, clipped to [0, 255]."""
    return np.clip(img.astype(np.int16) + b, 0, 255).astype(np.uint8)


inv = invert_image(scene)
brt = shift_brightness(scene, 60)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].imshow(scene, cmap='gray', vmin=0, vmax=255); ax[0].set_title("original"); ax[0].axis('off')
ax[1].imshow(inv, cmap='gray', vmin=0, vmax=255); ax[1].set_title("inverted"); ax[1].axis('off')
ax[2].imshow(brt, cmap='gray', vmin=0, vmax=255); ax[2].set_title("+60 brightness"); ax[2].axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# ✅ Self check — do not edit
assert invert_image(np.array([0, 128, 255], dtype=np.uint8)).tolist() == [255, 127, 0], \
    "invert_image is wrong: it must compute 255 - img"
assert shift_brightness(np.array([0, 250], dtype=np.uint8), 30).tolist() == [30, 255], \
    "shift_brightness must clip to [0, 255]"
print("✅ Task 1 assertions passed!")



### 🛠️ Task 2 — Colour channels 
A colour image has 3 channels. Create a **red** patch and a **green** patch on a
black background (remember: OpenCV stores **BGR**). Print the mean value of each
channel of the red patch.



In [ ]:
colour = np.zeros((120, 200, 3), dtype=np.uint8)
# OpenCV stores colours in BGR order.
cv2.rectangle(colour, (20, 20), (119, 79), (0, 0, 255), -1)
cv2.rectangle(colour, (130, 35), (184, 99), (0, 255, 0), -1)
plt.imshow(cv2.cvtColor(colour, cv2.COLOR_BGR2RGB))
plt.title("BGR image shown as RGB"); plt.axis('off'); plt.show()
red_patch = colour[20:80, 20:120]
print("mean B,G,R of the red region:", red_patch.reshape(-1, 3).mean(axis=0))


## Part 2 · Pinhole camera & perspective projection 

**Recap.** With the camera at the origin and the image plane at distance $f$ in front
of it, a 3-D point $(X,Y,Z)$ projects to

$$x = f\,\frac{X}{Z}, \qquad y = f\,\frac{Y}{Z}$$

This is *perspective* projection. In homogeneous coordinates the linear model is
$\lambda\,\tilde{\mathbf{x}} = \mathrm{diag}(f,f,1)\,[I\mid 0]\,\tilde{\mathbf{X}}$.

**Effects you should be able to explain:** objects shrink as $Z$ grows, parallel
lines converge to a vanishing point, and a real pinhole produces an *inverted*
image (we usually use a virtual plane at $+f$ to avoid flipping).



In [ ]:
def project_pinhole(points3d, f):
    """Project visible 3-D points with x=fX/Z, y=fY/Z."""
    points3d = np.asarray(points3d, dtype=np.float64)
    points3d = points3d[points3d[:, 2] > 0]
    return f * points3d[:, :2] / points3d[:, 2:3]


world = np.array([[1., 1., 5.], [1., 1., 10.], [0., 2., 8.], [-1., -1., 4.]])
f = 50.0
px = project_pinhole(world, f)
print("projected 2-D points:"); print(px)
near, far = px[0], px[1]
print("near (%.2f, %.2f)  far (%.2f, %.2f)" % (near[0], near[1], far[0], far[1]))


In [ ]:
# ✅ Self check — do not edit
_expected = np.array([[10., 10.], [5., 5.], [0., 12.5], [-12.5, -12.5]])
px = project_pinhole(world, f)
assert np.allclose(px, _expected), "project_pinhole output does not match hand calculation"
print("✅ Task 3 assertions passed — your projection is correct!")



### 📝 Task 3 — Conceptual questions
1. It is 2 times smaller: projected size is proportional to $1/Z$, so doubling depth halves image size.
2. Along each rail, $X$ grows more slowly relative to its increasing $Z$; therefore $fX/Z$ approaches the same limiting image coordinate for both rails.
3. Rays from the top and bottom of an object cross at the pinhole, so they reach opposite sides of the back wall.
4. $x=sX$, $y=sY$. It is reasonable when depth variation is small compared with camera distance (or with a long focal length / narrow field of view).


In [ ]:
def project_orthographic(points3d, s=1.0):
    """Orthographic projection: drop Z and scale X,Y."""
    return s * np.asarray(points3d, dtype=np.float64)[:, :2]

px_p = project_pinhole(world, 50.0)
px_o = project_orthographic(world, 50.0)
plt.figure(figsize=(10, 6))
plt.scatter(px_p[:, 0], -px_p[:, 1], c='red', label='perspective', s=90, marker='o')
plt.scatter(px_o[:, 0], -px_o[:, 1], c='blue', label='orthographic', s=60, marker='x')
plt.gca().set_aspect('equal'); plt.grid(alpha=.3); plt.legend(); plt.show()


## Part 3 · Light & Lambertian shading *(optional recap)*

**Recap.** For a Lambertian (matte) surface the reflected intensity is
$I = k_d \max(0,\, \hat N\cdot \hat L)$, i.e. bright when the surface normal
$\hat N$ points toward the light $\hat L$. This is the simplest model of how a 3-D
shape produces a shaded 2-D image — the link between *geometry* and *pixels*.

Implement the dot-product shading for a synthetic sphere (its normals are already
computed for you).



In [ ]:
r = 60
yy, xx = np.mgrid[0:150, 0:150]
xc, yc = 75, 75
d2 = (xx - xc) ** 2 + (yy - yc) ** 2
inside = d2 <= r * r
Nz = np.sqrt(np.clip(r * r - d2, 0, None))
N = np.zeros((150, 150, 3))
N[..., 0] = (xx - xc) / r; N[..., 1] = (yy - yc) / r; N[..., 2] = Nz / r
N[~inside] = 0.0

def lambertian_shade(N, L):
    """Return max(0, N dot L)."""
    L = np.asarray(L, dtype=np.float64); L /= np.linalg.norm(L)
    return np.clip(np.einsum('ijk,k->ij', N, L), 0, None)

L1 = np.array([0., 0., 1.]); L2 = np.array([1., .2, .8]); L2 /= np.linalg.norm(L2)
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for a, L, t in zip(ax, (L1, L2), ("light from front", "light from upper-right")):
    a.imshow(lambertian_shade(N, L), cmap='gray'); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()


## Part 4 · Camera geometry: extrinsic → intrinsic → pixels 

**Recap.** Going from a *world* point $\mathbf{X}_w$ to a *pixel* $\mathbf{x}$ needs two
steps:

1. **Extrinsic** (where is the camera in the world?): rotate then translate
   $\mathbf{X}_c = R\,\mathbf{X}_w + t$
2. **Intrinsic** (from camera coordinates to pixels):
   $K=\begin{bmatrix} f_x & 0 & c_x\\ 0 & f_y & c_y\\ 0&0&1\end{bmatrix}$, and in
   homogeneous coordinates $\lambda\,\tilde{\mathbf{x}} = K\,[R\mid t]\,\tilde{\mathbf{X}}_w$.

Write the two helper functions, then compare your result with OpenCV's `projectPoints`.



In [ ]:
def extrinsic_transform(Xw, R, t):
    """World -> camera coordinates: Xc = R @ Xw + t."""
    return (np.asarray(R) @ np.asarray(Xw, dtype=np.float64).T + np.asarray(t).reshape(3, 1)).T


def full_projection(Xw, K, R, t):
    """Project world points to pixel coordinates."""
    Xc = extrinsic_transform(Xw, R, t)
    homogeneous = (np.asarray(K) @ Xc.T).T
    return homogeneous[:, :2] / homogeneous[:, 2:3]

K = np.array([[600., 0, 320], [0, 600., 240], [0, 0, 1.]])
R = cv2.Rodrigues(np.array([.05, -.1, .2]))[0]
t = np.array([.1, -.2, 6.]).reshape(3, 1)
Xw = np.array([[0,0,0],[1,0,0],[0,1,0],[.5,.5,0]], dtype=np.float64)
mine = full_projection(Xw, K, R, t)
print(mine)


In [ ]:
# ✅ Self check — do not edit
Xw = np.array([[0, 0, 0], [1, 0, 0], [0, 1, 0], [0.5, 0.5, 0]], dtype=np.float64)
Rcv = cv2.Rodrigues(np.array([0.05, -0.1, 0.2]))[0]
tcv = np.array([0.1, -0.2, 6.0]).reshape(3, 1)
ref, _ = cv2.projectPoints(Xw, Rcv, tcv, K, np.zeros(5))
ref = ref.reshape(-1, 2)
assert np.allclose(full_projection(Xw, K, Rcv, tcv), ref, atol=1e-6), \
    "your full_projection does not match cv2.projectPoints"
print("✅ Task 5 assertions passed — matches OpenCV within 1e-6!")



## Part 5 · Camera calibration & distortion 

**Recap.** Calibration recovers $K$ (and distortion) from images of a known pattern
(checkerboard). Two classical routes:

* **DLT** — build the $2N\times 12$ matrix $A$ from the correspondences
  $u=\frac{P_{11}X+P_{12}Y+P_{13}Z+P_{14}}{P_{31}X+P_{32}Y+P_{33}Z+P_{34}}$ (and
  similarly for $v$), then $P$ is the last row of $V^\top$ in
  $\mathrm{SVD}(A)$.
* **`cv2.calibrateCamera`** — non-linear refinement of $K$, distortion and poses by
  minimising the **reprojection error**.

The first cell builds a **synthetic calibration rig**: it simulates 10 photos of a
checkerboard from random poses using a known ground-truth $K$. Run it, then complete
the two tasks.



In [ ]:
# ---- provided: synthetic calibration data (do not edit) -------------------
cb_size = (9, 6)          # inner corners
sq = 25.0                 # mm per square
objp = np.zeros((cb_size[0] * cb_size[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:cb_size[0], 0:cb_size[1]].T.reshape(-1, 2) * sq

K_truth = np.array([[800., 0, 320], [0, 800., 240], [0, 0, 1]], dtype=np.float32)
dist_truth = np.array([0.05, -0.1, 0.001, 0.001, 0.02], dtype=np.float32)  # k1,k2,p1,p2,k3

np.random.seed(0)
obj_points, img_points = [], []
for _ in range(10):
    rvec, _ = cv2.Rodrigues(np.array([np.random.uniform(-.6, .6),
                                      np.random.uniform(-.6, .6),
                                      np.random.uniform(-.3, .3)], np.float32))
    tvec = np.array([[np.random.uniform(-40, 40)],
                     [np.random.uniform(-40, 40)],
                     [np.random.uniform(350, 650)]], np.float32)
    pts, _ = cv2.projectPoints(objp, rvec, tvec, K_truth, dist_truth)
    img_points.append((pts.reshape(-1, 2) + np.random.normal(0, .2, (len(objp), 2))).astype(np.float32))
    obj_points.append(objp)

print(f"Simulated {len(img_points)} checkerboard views. Ground-truth K:\n{K_truth}")



In [ ]:
def dlt_calibrate(points_3d, points_2d):
    """Estimate P by DLT; use a homography for coplanar calibration targets."""
    X = np.asarray(points_3d, dtype=np.float64)
    x = np.asarray(points_2d, dtype=np.float64)
    # A planar checkerboard cannot constrain P's column normal to its plane.
    # Estimate its well-defined plane-to-image homography instead.
    if np.ptp(X[:, 2]) < 1e-12:
        A = []
        for (Xc, Yc, _), (u, v) in zip(X, x):
            A.extend(([Xc, Yc, 1, 0, 0, 0, -u*Xc, -u*Yc, -u],
                      [0, 0, 0, Xc, Yc, 1, -v*Xc, -v*Yc, -v]))
        _, _, Vt = np.linalg.svd(np.asarray(A))
        H = Vt[-1].reshape(3, 3)
        return np.column_stack((H[:, 0], H[:, 1], np.zeros(3), H[:, 2]))
    A = []
    for (Xc, Yc, Zc), (u, v) in zip(X, x):
        A.extend(([Xc,Yc,Zc,1,0,0,0,0,-u*Xc,-u*Yc,-u*Zc,-u],
                  [0,0,0,0,Xc,Yc,Zc,1,-v*Xc,-v*Yc,-v*Zc,-v]))
    _, _, Vt = np.linalg.svd(np.asarray(A))
    return Vt[-1].reshape(3, 4)

errs = []
for obj, img in zip(obj_points, img_points):
    P = dlt_calibrate(obj, img); P /= np.linalg.norm(P)
    hom = np.hstack([obj, np.ones((len(obj), 1))])
    proj = (P @ hom.T).T; proj = proj[:, :2] / proj[:, 2:3]
    errs.append(np.linalg.norm(proj - img, axis=1).mean())
print("P shape:", P.shape)
print(f"DLT mean reprojection error over all views: {np.mean(errs):.3f} px")


In [ ]:
# ---- TASK 7: use cv2.calibrateCamera on ALL views -------------------------
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(obj_points, img_points,
                                                  (640, 480), None, None)

print(f"✅ calibration RMS reprojection error = {ret:.4f} px")
print("Estimated K:\n", np.round(mtx, 1))
print("Ground-truth K:\n", K_truth)
rel = np.linalg.norm(mtx.astype(np.float64) - K_truth) / np.linalg.norm(K_truth)
print(f"Relative error on K = {rel:.5f}")
print("Estimated distortion:", np.round(dist.ravel(), 4))
print("Ground-truth distortion:", dist_truth)



### 📝 Task 8 — Interpreting the calibration
1. Multiple tilted views constrain focal length, principal point, pose, and distortion; one frontal view leaves important parameter ambiguities.
2. It is the RMS distance between observed and projected checkerboard corners; smaller is better, ideally well below one pixel for clean data.
3. $f_x,f_y$ are diagonal entries; $c_x,c_y$ are offsets in the last column. A centred 640×480 sensor has principal point near $(320,240)$.


In [ ]:
# ---- TASK 9 (extra): distortion correction demo ---------------------------
# Build a clean synthetic checkerboard image
h, w = 480, 640
board = np.full((h, w), 255, dtype=np.uint8)
s = 40
for i in range(0, h, s):
    for j in range(0, w, s):
        if (i // s + j // s) % 2 == 0:
            board[i:i + s, j:j + s] = 0

# strong barrel distortion coefficients for a *visible* effect
dist_barrel = np.array([0.6, -0.5, 0, 0, 0], dtype=np.float32)
distorted = cv2.undistort(board, K_truth, dist_barrel)          # apply distortion
corrected = cv2.undistort(distorted, mtx, -dist_barrel)          # undo it (approx.)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].imshow(board, cmap='gray');     ax[0].set_title("original");      ax[0].axis('off')
ax[1].imshow(distorted, cmap='gray'); ax[1].set_title("barrel-distorted"); ax[1].axis('off')
ax[2].imshow(corrected, cmap='gray'); ax[2].set_title("undistorted");    ax[2].axis('off')
plt.tight_layout(); plt.show()
print("Negative k1 -> barrel distortion; passing the negated coefficients removes it. ✅")



## ✅ Lab 1 — Reflection
I understand the projection pipeline best: transform world points to camera coordinates, then divide by depth and apply intrinsics.

A useful workshop question: how do we detect and reject inaccurate checkerboard corners before calibration?
